# Invecchiamento dei volti con PatchCore — UTKFace

**Studio non supervisionato di anomaly detection.** Usiamo PatchCore non per dire se un volto è "difettoso",
ma come *strumento di misura* della distanza dall'essere giovane. Costruiamo il memory bank (la "normalità")
usando **solo volti giovani**; poi facciamo passare volti di ogni età e misuriamo quanto si discostano.

**Domande di ricerca:**
- **B1** — Il punteggio di anomalia cresce con l'età? E dove si localizza la deviazione sul viso?
- **B3** — Invecchiare è più "anomalo" che ringiovanire? (deviazione simmetrica attorno a un'età di riferimento)
- **B4** — Quali regioni del viso sono più *invarianti* all'età?
- **A2** — Conviene comprimere la memoria con il **coreset greedy (k-center)** o con i **centroidi del K-Means**?

**Nota onesta:** PatchCore non "capisce l'età". Misura quanto la *rappresentazione* di un volto si allontana
da quella dei giovani; noi interpretiamo quella deviazione come segni dell'invecchiamento.

**Requisiti:** `torch`, `torchvision`, `scikit-learn`, `numpy`, `pandas`, `matplotlib`, `pillow`, `tqdm`.

## 0. Configurazione

In [ ]:
import os, glob, random, zipfile
import numpy as np
import pandas as pd
import torch, torchvision
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

# ---- Riproducibilità ----
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# ---- Percorsi (ADATTA QUI) ----
ZIP_PATH    = "archive.zip"          # nome del file zip UTKFace scaricato da Kaggle
EXTRACT_DIR = "data_utk"             # cartella di estrazione
DATA_DIR    = os.path.join(EXTRACT_DIR, "UTKFace")  # cartella con le immagini

# ---- Parametri dello studio ----
YOUNG_RANGE     = (20, 30)   # "normalità" per B1
MID_RANGE       = (40, 50)   # riferimento per B3 (simmetria)
N_BANK_IMAGES   = 500        # immagini di riferimento per il memory bank
PATCH_CAP       = 150_000    # tetto di patch prima del coreset (tempi gestibili)
CORESET_RATIO   = 0.02       # frazione di patch tenute nel coreset
N_SCORE_PER_BIN = 250        # immagini valutate per fascia d'età
IMG_BATCH       = 16
NUM_WORKERS     = 2          # su Windows metti 0 se hai problemi
AGG_TOPK        = 10         # punteggio immagine = media delle topK distanze di patch

AGE_BINS = [(1,9),(10,19),(20,29),(30,39),(40,49),(50,59),(60,69),(70,120)]

## 1. Caricamento e pulizia di UTKFace

I nomi file UTKFace hanno il formato `età_genere_etnia_data.jpg.chip.jpg`
(es. `25_0_0_2017...jpg` = 25 anni, maschio, bianco). Le etichette si leggono dal nome.

**Pulizia:** usiamo *solo* la cartella `UTKFace/` (lo zip contiene copie duplicate), scartiamo i ~3 file
con nome malformato e validiamo i range (età 1–116, genere 0/1, etnia 0–4).

In [ ]:
# Estrazione (se serve)
if not os.path.isdir(DATA_DIR):
    if os.path.exists(ZIP_PATH):
        print("Estrazione di", ZIP_PATH, "...")
        with zipfile.ZipFile(ZIP_PATH) as z:
            z.extractall(EXTRACT_DIR)
    else:
        print("ATTENZIONE: imposta ZIP_PATH o DATA_DIR correttamente.")

# La cartella UTKFace può essere annidata: la cerchiamo
if not os.path.isdir(DATA_DIR):
    cand = glob.glob(os.path.join(EXTRACT_DIR, "**", "UTKFace"), recursive=True)
    if cand:
        DATA_DIR = cand[0]
print("DATA_DIR =", DATA_DIR, "| esiste:", os.path.isdir(DATA_DIR))

In [ ]:
RACE_LABELS   = {0:"Bianco", 1:"Nero", 2:"Asiatico", 3:"Indiano", 4:"Altri"}
GENDER_LABELS = {0:"Maschio", 1:"Femmina"}

def parse_utkface(fname):
    """Estrae (age, gender, race) dal nome file UTKFace. None se malformato o fuori range."""
    parts = fname.split("_")
    if len(parts) < 4:
        return None
    try:
        age, gender, race = int(parts[0]), int(parts[1]), int(parts[2])
    except ValueError:
        return None
    if not (1 <= age <= 116):   return None
    if gender not in (0, 1):    return None
    if race not in (0,1,2,3,4): return None
    return age, gender, race

records, malformed = [], 0
for fname in sorted(os.listdir(DATA_DIR)):
    if not fname.lower().endswith(".jpg"):
        continue
    parsed = parse_utkface(fname)
    if parsed is None:
        malformed += 1
        continue
    age, gender, race = parsed
    records.append({"path": os.path.join(DATA_DIR, fname),
                    "age": age, "gender": gender, "race": race})

df = pd.DataFrame(records)
print(f"Immagini valide: {len(df)}  |  scartate (malformate): {malformed}")
df.head()

In [ ]:
# Distribuzioni (audit dei dati)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
df["age"].plot.hist(bins=40, ax=ax[0]); ax[0].set_title("Distribuzione età"); ax[0].set_xlabel("età")
df["gender"].map(GENDER_LABELS).value_counts().plot.bar(ax=ax[1]); ax[1].set_title("Genere")
df["race"].map(RACE_LABELS).value_counts().plot.bar(ax=ax[2]); ax[2].set_title("Etnia")
plt.tight_layout(); plt.show()
print(df["age"].describe().round(1))

## 2. Modello pre-addestrato ed estrattore di patch

Esattamente come nel tuo notebook PatchCore: **ResNet18 pre-addestrata su ImageNet** (congelata, nessun
addestramento). Estraiamo le patch dagli output di **layer2** (128×28×28) e **layer3** (256×14×14, riportato
a 28×28), concatenati lungo i canali → **384×28×28** → ogni immagine dà **784 patch** da 384 dimensioni.

In [ ]:
resnet = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
resnet.eval().to(DEVICE)
transform = torchvision.models.ResNet18_Weights.IMAGENET1K_V1.transforms()
print(transform)

In [ ]:
GRID, PATCH_DIM = 28, 384

@torch.no_grad()
def patch_extractor(resnet, imgs):
    """Estrae le patch da un minibatch (come nel notebook PatchCore).
    layer2 -> Bx128x28x28 ; layer3 -> Bx256x14x14 (interpolato a 28x28).
    Concatenati -> Bx384x28x28 -> (B*784)x384."""
    layer_outputs = []
    def hook(module, inputs, outputs):
        layer_outputs.append(outputs)
    h2 = resnet.get_submodule("layer2").register_forward_hook(hook)
    h3 = resnet.get_submodule("layer3").register_forward_hook(hook)
    resnet.eval()
    resnet(imgs.to(DEVICE))
    h2.remove(); h3.remove()
    l2, l3 = layer_outputs
    l3 = F.interpolate(l3, size=(GRID, GRID), mode="bilinear", align_corners=False)
    patches = torch.cat([l2, l3], dim=1)                      # B x 384 x 28 x 28
    return patches.permute(0, 2, 3, 1).reshape(-1, patches.shape[1])  # (B*784) x 384

## 3. Dataset UTKFace

In [ ]:
class UTKFaceDataset(Dataset):
    def __init__(self, frame, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        x = torch.tensor(np.array(img)).permute(2, 0, 1)   # uint8 CHW
        if self.transform:
            x = self.transform(x)
        return x, int(row["age"]), int(row["gender"]), int(row["race"])

## 4. Memory bank sui volti giovani (B1)

Campioniamo `N_BANK_IMAGES` volti **giovani** (20–30 anni) e ne estraiamo tutte le patch: è la "memoria del
normale". Limitiamo il numero di immagini per tenere il banco gestibile (PatchCore funziona bene anche con
pochi esempi normali — è il regime *cold-start*).

In [ ]:
def sample_by_age(frame, age_range, n, seed=SEED):
    lo, hi = age_range
    sub = frame[(frame.age >= lo) & (frame.age <= hi)]
    return sub.sample(n=min(n, len(sub)), random_state=seed).reset_index(drop=True)

@torch.no_grad()
def build_memory_bank(frame_subset):
    """Estrae e concatena le patch da un sottoinsieme di volti -> memory bank (su CPU)."""
    ds = UTKFaceDataset(frame_subset, transform=transform)
    loader = DataLoader(ds, batch_size=IMG_BATCH, shuffle=False, num_workers=NUM_WORKERS)
    chunks = []
    for imgs, *_ in tqdm(loader, desc="Memory bank"):
        chunks.append(patch_extractor(resnet, imgs).cpu())
    return torch.cat(chunks, dim=0)

young_df = sample_by_age(df, YOUNG_RANGE, N_BANK_IMAGES)
print("Volti giovani per il memory bank:", len(young_df))
memory_bank = build_memory_bank(young_df)
print("Memory bank:", tuple(memory_bank.shape))

## 5. Coreset greedy (k-center)

Dal memory bank si estrae un campione di patch **il più diversificato possibile**: si scelgono iterativamente
i punti più lontani da quelli già scelti (come nel tuo notebook). Prima mettiamo un tetto al numero di patch
per tenere i tempi ragionevoli.

In [ ]:
def subsample_patches(bank, cap, seed=SEED):
    if len(bank) <= cap:
        return bank
    g = torch.Generator().manual_seed(seed)
    idx = torch.randperm(len(bank), generator=g)[:cap]
    return bank[idx]

@torch.no_grad()
def greedy_coreset(features, ratio=CORESET_RATIO, seed=SEED):
    """Coreset greedy (k-center): seleziona i punti che massimizzano la copertura dello spazio."""
    feats = features.to(DEVICE).float()
    n_total  = feats.shape[0]
    n_select = max(1, int(n_total * ratio))
    g = torch.Generator(device="cpu").manual_seed(seed)
    selected  = [int(torch.randint(n_total, (1,), generator=g).item())]
    min_dists = torch.full((n_total,), float("inf"), device=DEVICE)
    for _ in tqdm(range(n_select - 1), desc="Coreset"):
        last      = feats[selected[-1]].unsqueeze(0)
        dists     = torch.norm(feats - last, dim=1)
        min_dists = torch.minimum(min_dists, dists)
        selected.append(int(min_dists.argmax().item()))
    return features[selected]

bank_sub = subsample_patches(memory_bank, PATCH_CAP)
coreset  = greedy_coreset(bank_sub).to(DEVICE)
print("Coreset:", tuple(coreset.shape))

## 6. Punteggio di anomalia e mappa di localizzazione

Per ogni patch di un volto cerchiamo il **vicino più prossimo** nel coreset. Le 784 distanze formano la
**mappa 28×28** (dove sta l'anomalia); il punteggio dell'immagine è la media delle `AGG_TOPK` distanze più alte.

In [ ]:
@torch.no_grad()
def score_image_patches(patches_one, memory, agg_topk=AGG_TOPK):
    """patches_one: (784,384) di una immagine. memory: (M,384) sul device.
    Ritorna (punteggio_immagine, mappa 28x28)."""
    d = torch.cdist(patches_one, memory)            # (784, M)
    mind = d.min(dim=1).values                      # (784,)
    score = mind.topk(min(agg_topk, mind.numel())).values.mean().item()
    amap = mind.view(GRID, GRID).cpu().numpy()
    return score, amap

@torch.no_grad()
def score_frame(frame_subset, memory, want_maps=False):
    ds = UTKFaceDataset(frame_subset, transform=transform)
    loader = DataLoader(ds, batch_size=IMG_BATCH, shuffle=False, num_workers=NUM_WORKERS)
    memory = memory.to(DEVICE)
    scores, ages, genders, races, maps = [], [], [], [], []
    for imgs, a, g_, r in loader:
        p = patch_extractor(resnet, imgs).view(imgs.shape[0], GRID*GRID, PATCH_DIM)
        for i in range(imgs.shape[0]):
            s, m = score_image_patches(p[i], memory)
            scores.append(s)
            if want_maps: maps.append(m)
        ages += list(a.numpy()); genders += list(g_.numpy()); races += list(r.numpy())
    out = pd.DataFrame({"age": ages, "gender": genders, "race": races, "score": scores})
    return (out, maps) if want_maps else out

## 7. B1 — Il punteggio di anomalia cresce con l'età?

Valutiamo volti di tutte le età (esclusi quelli usati per il banco), campionando un numero simile per fascia,
e tracciamo punteggio vs età.

In [ ]:
bank_paths = set(young_df["path"])
score_pool = df[~df["path"].isin(bank_paths)]

parts = []
for lo, hi in AGE_BINS:
    sub = score_pool[(score_pool.age >= lo) & (score_pool.age <= hi)]
    if len(sub) == 0: continue
    parts.append(sub.sample(n=min(N_SCORE_PER_BIN, len(sub)), random_state=SEED))
score_df_input = pd.concat(parts).reset_index(drop=True)
print("Volti da valutare:", len(score_df_input))

res_b1 = score_frame(score_df_input, coreset)

m = res_b1.groupby((res_b1["age"] // 5) * 5)["score"].mean()
plt.figure(figsize=(8, 5))
plt.scatter(res_b1["age"], res_b1["score"], s=6, alpha=0.25)
plt.plot(m.index, m.values, color="red", marker="o", label="media ogni 5 anni")
plt.axvspan(YOUNG_RANGE[0], YOUNG_RANGE[1], color="green", alpha=0.12, label="riferimento (giovani)")
plt.xlabel("Età"); plt.ylabel("Punteggio di anomalia"); plt.legend()
plt.title("B1 — Anomalia vs età"); plt.show()

rho = np.corrcoef(res_b1["age"], res_b1["score"])[0, 1]
print(f"Correlazione (Pearson) età-punteggio: {rho:.3f}")

## 8. B1 — Dove si localizza la deviazione (mappe medie per fascia d'età)

Aggreghiamo le mappe di anomalia per fascia d'età. *Atteso:* l'anomalia si concentra dove l'età "si vede".
Questa parte è in buona misura confermativa — il valore concreto sta nelle curve (B1), nella simmetria (B3),
nella tabella per regione (B4) e nel confronto A2.

In [ ]:
def averaged_maps_by_band(frame, memory, bands, per_band=60):
    out = {}
    for lo, hi in bands:
        sub = frame[(frame.age >= lo) & (frame.age <= hi)]
        if len(sub) == 0: continue
        sub = sub.sample(n=min(per_band, len(sub)), random_state=SEED)
        _, maps = score_frame(sub, memory, want_maps=True)
        out[(lo, hi)] = np.mean(np.stack(maps, 0), 0)
    return out

bands_show = [(20,29),(40,49),(60,69),(70,120)]
amaps = averaged_maps_by_band(score_pool, coreset, bands_show, per_band=60)

vmax = max(m.max() for m in amaps.values())
fig, ax = plt.subplots(1, len(amaps), figsize=(4*len(amaps), 4))
for k, (band, m) in enumerate(amaps.items()):
    im = ax[k].imshow(m, cmap="inferno", vmin=0, vmax=vmax)
    ax[k].set_title(f"{band[0]}-{band[1]} anni"); ax[k].axis("off")
fig.colorbar(im, ax=ax, fraction=0.02)
plt.suptitle("B1 — Mappa media dell'anomalia per fascia d'età"); plt.show()

## 9. B3 — Invecchiare è più anomalo che ringiovanire?

Ricostruiamo il banco su un riferimento di **mezza età (40–50)** e guardiamo se il punteggio cresce in modo
*simmetrico* verso i giovani e verso gli anziani.

In [ ]:
mid_df      = sample_by_age(df, MID_RANGE, N_BANK_IMAGES)
memory_mid  = build_memory_bank(mid_df)
coreset_mid = greedy_coreset(subsample_patches(memory_mid, PATCH_CAP)).to(DEVICE)

mid_paths = set(mid_df["path"])
pool_mid  = df[~df["path"].isin(mid_paths)]
parts = []
for lo, hi in AGE_BINS:
    sub = pool_mid[(pool_mid.age >= lo) & (pool_mid.age <= hi)]
    if len(sub) == 0: continue
    parts.append(sub.sample(n=min(N_SCORE_PER_BIN, len(sub)), random_state=SEED))
res_b3 = score_frame(pd.concat(parts).reset_index(drop=True), coreset_mid)

m = res_b3.groupby((res_b3["age"] // 5) * 5)["score"].mean()
plt.figure(figsize=(8, 5))
plt.plot(m.index, m.values, marker="o")
plt.axvspan(MID_RANGE[0], MID_RANGE[1], color="orange", alpha=0.15, label="riferimento (40-50)")
plt.xlabel("Età"); plt.ylabel("Punteggio anomalia"); plt.legend()
plt.title("B3 — Deviazione attorno all'età di riferimento"); plt.show()

younger = res_b3[res_b3.age < MID_RANGE[0]]["score"].mean()
older   = res_b3[res_b3.age > MID_RANGE[1]]["score"].mean()
print(f"Punteggio medio più giovani del riferimento: {younger:.3f}")
print(f"Punteggio medio più anziani del riferimento: {older:.3f}")
print("Invecchiare è più anomalo che ringiovanire?", "SÌ" if older > younger else "NO")

## 10. B4 — Quali parti del viso sono invarianti all'età?

I volti UTKFace sono allineati, quindi possiamo associare bande della griglia 28×28 a regioni del viso
(approssimative). Misuriamo l'anomalia media per regione lungo le fasce d'età: la regione che cresce **meno**
è la più *invariante* all'età.

In [ ]:
REGIONS = {
    "Fronte": (slice(2, 9),   slice(6, 22)),
    "Occhi":  (slice(9, 14),  slice(4, 24)),
    "Naso":   (slice(13, 19), slice(10, 18)),
    "Bocca":  (slice(19, 24), slice(8, 20)),
    "Mento":  (slice(23, 28), slice(8, 20)),
}
bands_b4 = [(20,29),(30,39),(40,49),(50,59),(60,69),(70,120)]
amaps_b4 = averaged_maps_by_band(score_pool, coreset, bands_b4, per_band=80)

rows = []
for (lo, hi), m in amaps_b4.items():
    rec = {"fascia": f"{lo}-{hi}"}
    for name, (rs, cs) in REGIONS.items():
        rec[name] = float(m[rs, cs].mean())
    rows.append(rec)
region_df = pd.DataFrame(rows).set_index("fascia")
print(region_df.round(3))

growth = (region_df.iloc[-1] / region_df.iloc[0]).sort_values()
print("\nCrescita (ultima/prima fascia) per regione (più basso = più invariante):")
print(growth.round(3))

region_df.plot(marker="o", figsize=(8, 5), title="B4 — Anomalia media per regione vs età")
plt.xlabel("Fascia d'età"); plt.ylabel("Anomalia media"); plt.show()

## 11. A2 — Coreset (k-center) vs centroidi K-Means come memoria

Confrontiamo i due modi di comprimere la memoria su un **compito concreto**: distinguere volti **giovani (20–30)**
da **anziani (60+)** trattando gli anziani come "anomalia". A parità di numero di punti, misuriamo l'**AUROC**.

In [ ]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import roc_auc_score

M = coreset.shape[0]
bank_np = bank_sub.cpu().numpy()
print(f"K-Means: {M} centroidi su {len(bank_np)} patch...")
km = MiniBatchKMeans(n_clusters=M, random_state=SEED, batch_size=4096, n_init=3).fit(bank_np)
memory_km = torch.tensor(km.cluster_centers_, dtype=torch.float32)

eval_young = sample_by_age(score_pool, (20, 30), 300)
eval_old   = sample_by_age(score_pool, (60, 120), 300)
eval_df = pd.concat([eval_young, eval_old]).reset_index(drop=True)
y_true = (eval_df["age"] >= 60).astype(int).values

res_core = score_frame(eval_df, coreset)
res_km   = score_frame(eval_df, memory_km)
auc_core = roc_auc_score(y_true, res_core["score"].values)
auc_km   = roc_auc_score(y_true, res_km["score"].values)
print(f"AUROC (giovani vs anziani)   Coreset k-center: {auc_core:.3f}   |   K-Means: {auc_km:.3f}")

## 12. (Extra) Copertura del coreset nello spazio delle patch — t-SNE

Visualizziamo, con t-SNE, come i punti del coreset (rosso) coprono lo spazio delle patch (campione grigio).

In [ ]:
from sklearn.manifold import TSNE

sub_idx  = np.random.RandomState(SEED).choice(len(bank_sub), size=min(3000, len(bank_sub)), replace=False)
emb_all  = bank_sub[sub_idx].cpu().numpy()
emb_core = coreset.cpu().numpy()
ts = TSNE(n_components=2, random_state=SEED, init="pca",
          perplexity=30).fit_transform(np.vstack([emb_all, emb_core]))

plt.figure(figsize=(7, 6))
plt.scatter(ts[:len(emb_all), 0], ts[:len(emb_all), 1], s=4, alpha=0.3, label="patch (campione)")
plt.scatter(ts[len(emb_all):, 0], ts[len(emb_all):, 1], s=12, c="red", label="coreset")
plt.legend(); plt.title("Copertura del coreset (t-SNE)"); plt.axis("off"); plt.show()

## 13. Conclusioni

Compila qui i tuoi risultati con i numeri ottenuti:

- **B1** — Correlazione età–punteggio = `…`. Il punteggio cresce con l'età? La curva è lineare o accelera?
- **B1 (localizzazione)** — Le regioni che si "accendono" con l'età sono `…` (atteso: occhi/fronte/pelle).
- **B3** — Più giovani: `…` · più anziani: `…` → invecchiare è più anomalo che ringiovanire? `…`
- **B4** — Regione più invariante all'età: `…` · regione che cambia di più: `…`
- **A2** — AUROC coreset = `…` vs K-Means = `…` → quale memoria comprime meglio?

**Argomenti del programma coperti:** anomaly detection (PatchCore), memory bank + coreset (k-center),
K-Means (A2), distanze nello spazio delle feature (nearest neighbor), t-SNE (visualizzazione).

**Limite/onestà:** PatchCore misura la deviazione della *rappresentazione* rispetto al gruppo di riferimento;
l'interpretazione come "invecchiamento" va argomentata, non data per scontata. Possibili confondenti
(illuminazione, qualità foto, distribuzione etnica per età) andrebbero discussi.